# M4: DFT

Die DFT soll selbst programmiert und mit den Handrechnungen von Blatt 05 verglichen werden.

**Konvention** (wie in Vorlesung und Übung): der Faktor $1/N$ steht in der Hintransformation,

$$X(k) = \frac{1}{N} \sum_{n=0}^{N-1} x(n)\, e^{-j 2\pi k n / N}, \qquad x(n) = \sum_{k=0}^{N-1} X(k)\, e^{j 2\pi k n / N}.$$

`numpy` und Matlab setzen $1/N$ in die Rücktransformation. In unserer Konvention gilt also `X = np.fft.fft(x) / N` und `x = N * np.fft.ifft(X)`.

In [ ]:
from timeit import timeit

import numpy as np
from matplotlib import pyplot as plt
from plotly import graph_objects as go
from plotly.subplots import make_subplots

## a) 1D-DFT

Zwei Varianten: direkt nach der Formel mit zwei Schleifen und als Matrix-Vektor-Produkt mit der Drehfaktormatrix $W_{kn} = e^{-j 2\pi k n / N}$.

In [ ]:
def dft_loop(x):
    N = len(x)
    X = np.zeros(N, dtype=complex)
    for k in range(N):
        for n in range(N):
            X[k] += x[n] * np.exp(-2j * np.pi * k * n / N)
    return X / N


def dft(x):
    n = np.arange(len(x))
    W = np.exp(-2j * np.pi * np.outer(n, n) / len(x))  # W[k, n] = W_N^(k n)
    return W @ x / len(x)


x = np.random.default_rng(0).normal(size=16)
print(np.allclose(dft_loop(x), np.fft.fft(x) / len(x)), np.allclose(dft(x), np.fft.fft(x) / len(x)))

### Probe mit A13

a) $x(n) = |\cos(n\pi/4)|$, $N = 4$: erwartet $X = \{(1+\sqrt 2)/4,\ 1/4,\ (1-\sqrt 2)/4,\ 1/4\}$.

b) Rechteckfolge mit $N = 14$: $X(0) = 1/2$, $X(k) = 0$ für gerade $k \neq 0$ und $X(k) = \frac{1}{14}\left(1 - j \cot\frac{\pi k}{14}\right)$ für ungerade $k$.

In [ ]:
x13a = np.abs(np.cos(np.arange(4) * np.pi / 4))
X13a = dft(x13a)
print("A13 a:", np.round(X13a.real, 4), np.allclose(X13a, [(1 + np.sqrt(2)) / 4, 1 / 4, (1 - np.sqrt(2)) / 4, 1 / 4]))

x13b = np.r_[np.ones(7), np.zeros(7)]
X13b = dft(x13b)
odd = np.arange(1, 14, 2)
print("A13 b: gerade k null:", np.allclose(X13b[2::2], 0))
print("A13 b: Formel für ungerade k:", np.allclose(X13b[odd], (1 - 1j / np.tan(np.pi * odd / 14)) / 14))
print("|X(k)| ungerade:", np.round(np.abs(X13b[odd]), 4))

### Probe mit A14

$\cos(2\pi f_0 t)$ mit $f_0 = 1$ Hz, abgetastet mit $T_s = 0.25$ s, also $x(n) = \cos(n\pi/2)$, für $N = 4, 8, 9$. Die Frequenzachse ist $f_k = k f_s / N$.

In [ ]:
t_s = 0.25
f_s = 1 / t_s
x14 = {N: np.cos(np.arange(N) * np.pi / 2) for N in (4, 8, 9)}
X14 = {N: dft(x) for N, x in x14.items()}
for N, X in X14.items():
    print(f"N = {N}, Δf = {f_s / N:.3f} Hz")
    for k, value in enumerate(X):
        print(f"  k = {k}  f = {k * f_s / N:5.2f} Hz  X = {value.real:+.4f} {value.imag:+.4f}j  |X| = {abs(value):.4f}")

Bei $N = 4$ und $N = 8$ liegt $f_0$ auf dem Raster: zwei Linien der Höhe $1/2$ bei $f_0$ und $f_s - f_0$ (negative Frequenz). Bei $N = 9$ liegt $f_0 = 2.25\,\Delta f$ zwischen zwei Rasterpunkten; die periodische Fortsetzung hat einen Knick, und alle $X_3(k)$ sind ungleich null (Leckeffekt).

## b) 1D-IDFT, Probe mit A15

In [ ]:
def idft(X):
    n = np.arange(len(X))
    return np.exp(2j * np.pi * np.outer(n, n) / len(X)) @ X


X15 = np.array([1, 1 - 1j * np.sqrt(2), 1, 1 + 1j * np.sqrt(2)])
x15 = idft(X15)
print("x(n) =", np.round(x15, 4))
print("max |Im x|:", np.max(np.abs(x15.imag)))
print("wie numpy:", np.allclose(x15, len(X15) * np.fft.ifft(X15)), " Hin und zurück:", np.allclose(dft(x15), X15))
x15 = x15.real  # Imaginärteil ist nur Rundungsrauschen

$x(n) = \{4, 2\sqrt 2, 0, -2\sqrt 2\}$. Zerlegt in die IDFT-Anteile ist $x(n) = 1 + (-1)^n + 2\cos(\pi n/2) + 2\sqrt 2 \sin(\pi n/2)$; diese Summe, kontinuierlich gezeichnet, geht durch alle Werte (Abbildung unten).

## c) Zusatz: 2D-DFT direkt und separiert

Direkt: für jedes $(k_y, k_x)$ eine Summe über alle Pixel, zusammen $N^4$ Multiplikationen bei $N \times N$ Pixeln. Separiert: erst die 1D-DFT jeder Zeile, dann jeder Spalte, zusammen $2N^3$.

In [ ]:
def dft2_direct(s):
    n_y, n_x = s.shape
    ny, nx = np.arange(n_y)[:, None], np.arange(n_x)[None, :]
    S = np.zeros(s.shape, dtype=complex)
    for ky in range(n_y):
        for kx in range(n_x):
            S[ky, kx] = np.sum(s * np.exp(-2j * np.pi * (kx * nx / n_x + ky * ny / n_y)))
    return S / s.size


def dft2_separable(s):
    rows = np.array([dft(row) for row in s])  # 1D-DFT jeder Zeile
    return np.array([dft(col) for col in rows.T]).T  # dann jeder Spalte


s = np.array([[0, 1, 0, 0], [1, 1, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0]])
S = dft2_separable(s)
print("16 S =\n", np.round(16 * S, 4))
print("direkt = separiert = numpy:", np.allclose(dft2_direct(s), S), np.allclose(S, np.fft.fft2(s) / s.size))

rng = np.random.default_rng(1)
print("\n  N   direkt [ms]  separiert [ms]  Faktor  (N/2)")
for N in (4, 16, 32, 64):
    image = rng.random((N, N))
    repeats = 3 if N > 16 else 20
    t_direct = timeit(lambda image=image: dft2_direct(image), number=repeats) / repeats
    t_sep = timeit(lambda image=image: dft2_separable(image), number=repeats) / repeats
    print(f"{N:3d}  {t_direct * 1e3:11.3f}  {t_sep * 1e3:14.3f}  {t_direct / t_sep:6.1f}  ({N // 2})")

Die Messung folgt der Vorhersage $N/2$ nur grob: `numpy` rechnet die inneren Summen vektorisiert, und bei großen $N$ kosten auch der Aufbau der Drehfaktormatrizen und die Speicherzugriffe Zeit. Der Trend ist aber eindeutig: Die separierte Variante wird mit wachsendem $N$ immer vorteilhafter.

## Lösungsabbildung

Oben die drei abgetasteten Signale aus A14, in der Mitte ihre DFT über der Frequenz, unten A13 a, A13 b und A15. Einmal interaktiv mit Plotly, einmal mit matplotlib.

In [ ]:
t = np.linspace(0, 2.25, 400)
k13b = np.arange(14)
n15 = np.arange(12)
t15 = np.linspace(-0.3, 11.3, 500)
interp15 = 1 + np.cos(np.pi * t15) + 2 * np.cos(np.pi * t15 / 2) + 2 * np.sqrt(2) * np.sin(np.pi * t15 / 2)
titles = [f"x{i}(n), N = {N}" for i, N in enumerate(x14, start=1)]
titles += [f"|X{i}(f)|, N = {N}" for i, N in enumerate(x14, start=1)]
titles += ["A13 a: X(k)", "A13 b: |X(k)|", "A15: x(n)"]


def stems(fig, x, y, row, col, color):
    fig.add_trace(go.Scatter(x=x, y=y, mode="markers", marker_color=color, showlegend=False), row=row, col=col)
    for xx, yy in zip(x, y, strict=True):
        fig.add_shape(type="line", x0=xx, x1=xx, y0=0, y1=yy, line=dict(color=color, width=1), row=row, col=col)


fig = make_subplots(rows=3, cols=3, subplot_titles=titles, vertical_spacing=0.1)
for col, (N, x) in enumerate(x14.items(), start=1):
    fig.add_trace(go.Scatter(x=t, y=np.cos(2 * np.pi * t), line=dict(color="lightgray"), showlegend=False), row=1, col=col)
    stems(fig, np.arange(N) * t_s, x, 1, col, "royalblue")
    stems(fig, np.arange(N) * f_s / N, np.abs(X14[N]), 2, col, "crimson")
    fig.add_vline(x=1, line=dict(color="green", dash="dash", width=1), row=2, col=col)
    fig.add_vline(x=3, line=dict(color="green", dash="dash", width=1), row=2, col=col)
    fig.update_xaxes(title_text="t in s", row=1, col=col)
    fig.update_xaxes(title_text="f in Hz", range=[-0.2, 4.2], row=2, col=col)
stems(fig, np.arange(4), X13a.real, 3, 1, "crimson")
stems(fig, k13b, np.abs(X13b), 3, 2, "crimson")
fig.add_trace(go.Scatter(x=t15, y=interp15, line=dict(color="salmon"), showlegend=False), row=3, col=3)
stems(fig, n15, x15[n15 % 4], 3, 3, "royalblue")
for col, label in enumerate(["k", "k", "n"], start=1):
    fig.update_xaxes(title_text=label, row=3, col=col)
fig.update_layout(height=850, title="Blatt 05: DFT (Konvention mit 1/N in der Hintransformation)")
fig.show()

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(13, 9), layout="constrained")
for col, (N, x) in enumerate(x14.items()):
    axes[0, col].plot(t, np.cos(2 * np.pi * t), color="lightgray")
    axes[0, col].stem(np.arange(N) * t_s, x, basefmt="k-")
    axes[0, col].set(title=titles[col], xlabel="t in s")
    axes[1, col].stem(np.arange(N) * f_s / N, np.abs(X14[N]), linefmt="C3-", markerfmt="C3o", basefmt="k-")
    for f in (1, 3):
        axes[1, col].axvline(f, color="green", linestyle="--", linewidth=0.8)
    axes[1, col].set(title=titles[3 + col], xlabel="f in Hz", xlim=(-0.2, 4.2), ylim=(0, 0.6))
axes[2, 0].stem(np.arange(4), X13a.real, linefmt="C3-", markerfmt="C3o", basefmt="k-")
axes[2, 1].stem(k13b, np.abs(X13b), linefmt="C3-", markerfmt="C3o", basefmt="k-")
axes[2, 2].plot(t15, interp15, color="salmon")
axes[2, 2].stem(n15, x15[n15 % 4], basefmt="k-")
for ax, title, label in zip(axes[2], titles[6:], ["k", "k", "n"], strict=True):
    ax.set(title=title, xlabel=label)
plt.show()